## load the model

In [1]:
import torch
from src.assignment1.model import UNet

device = "cuda" if torch.cuda.is_available() else "cpu"

model = UNet().to(device)

ckpt = torch.load(
    "C:/Users/hanqingx/projects/diffusion-models/checkpoints/ddpm_epoch_49.pth",
    map_location=device
)

model.load_state_dict(ckpt["ema"])
model.eval()

UNet(
  (time_mlp): TimeEmbedding(
    (mlp): Sequential(
      (0): Linear(in_features=64, out_features=256, bias=True)
      (1): SiLU()
      (2): Linear(in_features=256, out_features=64, bias=True)
    )
  )
  (init_conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (down1): ModuleList(
    (0): ResidualBlock(
      (time_mlp): Sequential(
        (0): SiLU()
        (1): Linear(in_features=64, out_features=32, bias=True)
      )
      (conv1): Sequential(
        (0): GroupNorm(8, 32, eps=1e-05, affine=True)
        (1): SiLU()
        (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
      (conv2): Sequential(
        (0): GroupNorm(8, 32, eps=1e-05, affine=True)
        (1): SiLU()
        (2): Dropout(p=0.1, inplace=False)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      )
      (residual_conv): Identity()
    )
    (1): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  )
 

## 1000 samples

In [2]:
from src.assignment1.sample import sample

T = 300

beta = torch.linspace(1e-4, 0.02, T).to(device)
alpha = 1.0 - beta
alpha_cumprod = torch.cumprod(alpha, dim=0)

images = sample(model, 1000, 1, 28, T, alpha, alpha_cumprod, beta, device)

## load the classifier

In [3]:
from src.assignment1.classifier.helper import load_classifier
import torch.nn.functional as F

my_model = load_classifier()

Successfully loaded model from checkpoints/classifier_mnist_resnet.pth


In [4]:
images.size()

torch.Size([1000, 1, 28, 28])

## inception score

In [7]:
@torch.no_grad()
def get_probs_from_classifier(clf, images, device, batch_size=128):
    clf = clf.to(device).eval()
    probs_list = []
    for i in range(0, images.size(0), batch_size):
        x = images[i:i+batch_size].to(device)   # [B,1,28,28] in [0,1]
        x = x * 2.0 - 1.0                       # -> [-1,1], match classifier training
        logits = clf(x)
        probs = F.softmax(logits, dim=1)
        probs_list.append(probs)
    return torch.cat(probs_list, dim=0)


def inception_score_from_probs(probs, splits=10, eps=1e-16):
    """
    probs: torch.Tensor [N, K], each row sums to 1
    IS = exp(E_x KL(p(y|x) || p(y)))
    returns (mean, std) across splits
    """
    probs = probs.detach().cpu()
    N = probs.size(0)
    K = probs.size(1)

    assert N >= splits, "Need N >= splits"
    assert N % splits == 0, f"N ({N}) must be divisible by splits ({splits})"

    split_size = N // splits
    scores = []

    for s in range(splits):
        part = probs[s*split_size:(s+1)*split_size]         # [split_size, K]
        py = part.mean(dim=0, keepdim=True)                 # [1, K]
        # KL(part || py) for each sample
        kl = part * (torch.log(part + eps) - torch.log(py + eps))  # [split_size, K]
        kl = kl.sum(dim=1).mean()                           # scalar
        scores.append(torch.exp(kl).item())

    scores_t = torch.tensor(scores)
    return scores_t.mean().item(), scores_t.std(unbiased=False).item()

def compute_is_for_ddpm_samples(
    images,                           # [1000,1,28,28] in [0,1]
    clf,
    device=None,
    batch_size=128,
    splits=10
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    probs = get_probs_from_classifier(clf, images, device=device, batch_size=batch_size)

    is_mean, is_std = inception_score_from_probs(probs, splits=splits)
    print(f"Inception Score (using your ResNetMini classifier): {is_mean:.4f} ± {is_std:.4f}")

    return is_mean, is_std, probs

In [8]:
is_mean, is_std, probs = compute_is_for_ddpm_samples(
    images=images,
    clf=my_model,
    device=device,
    batch_size=128,
    splits=10
)

Inception Score (using your ResNetMini classifier): 7.7934 ± 0.3447


## FID

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

class ResNetMiniFeatureExtractor(torch.nn.Module):
    def __init__(self, clf):
        super().__init__()
        self.conv1 = clf.conv1
        self.bn1 = clf.bn1
        self.layer1 = clf.layer1
        self.layer2 = clf.layer2
        self.layer3 = clf.layer3
        # without the linear layer

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, out.size()[2:])  # [B,10,1,1]
        out = out.view(out.size(0), -1)          # [B,10]
        return out

@torch.no_grad()
def extract_features_from_loader(feature_model, dataloader, device, max_images=None):
    feature_model.eval()
    feats = []
    seen = 0
    for x, _ in dataloader:
        x = x.to(device)                 # already [-1,1] due to get_dataloader()
        f = feature_model(x)             # [B,d]
        feats.append(f.detach().cpu())
        seen += x.size(0)
        if max_images is not None and seen >= max_images:
            break
    feats = torch.cat(feats, dim=0)
    if max_images is not None:
        feats = feats[:max_images]
    return feats.numpy()

@torch.no_grad()
def extract_features_from_generated(feature_model, images_01, device, batch_size=128):
    feature_model.eval()
    feats = []
    for i in range(0, images_01.size(0), batch_size):
        x = images_01[i:i+batch_size].to(device)  # [B,1,28,28] in [0,1]
        x = x * 2.0 - 1.0                         # -> [-1,1]
        f = feature_model(x)
        feats.append(f.detach().cpu())
    feats = torch.cat(feats, dim=0)
    return feats.numpy()

# mu and sigma for FID
def compute_stats(feats_np):
    mu = np.mean(feats_np, axis=0)
    sigma = np.cov(feats_np, rowvar=False)
    return mu, sigma

# FID
def fid_from_stats(mu1, sigma1, mu2, sigma2, eps=1e-6):
    diff = mu1 - mu2

    try:
        from scipy import linalg
        covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)

        if not np.isfinite(covmean).all():
            offset = np.eye(sigma1.shape[0]) * eps
            covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

        if np.iscomplexobj(covmean):
            covmean = covmean.real

    except Exception:
        w, v = np.linalg.eig(sigma1.dot(sigma2))
        w = np.maximum(w.real, 0)
        covmean = (v @ np.diag(np.sqrt(w)) @ np.linalg.inv(v)).real

    fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2.0 * np.trace(covmean)
    return float(fid)

# compute FID using your ResNetMini classifier as the feature extractor
def compute_fid_resnetmini(images_01, classifier, real_loader, device=None,
                           gen_batch_size=128, n_real=10000):
    
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    classifier = classifier.to(device).eval()
    feat_model = ResNetMiniFeatureExtractor(classifier).to(device).eval()

    # Real features (already [-1,1])
    real_feats = extract_features_from_loader(feat_model, real_loader, device, max_images=n_real)

    # Generated features (convert [0,1] -> [-1,1])
    gen_feats = extract_features_from_generated(feat_model, images_01, device, batch_size=gen_batch_size)

    mu_r, sig_r = compute_stats(real_feats)
    mu_g, sig_g = compute_stats(gen_feats)

    fid_value = fid_from_stats(mu_r, sig_r, mu_g, sig_g)
    return fid_value

In [ ]:
from assignment1.data import get_dataloader 
real_loader = get_dataloader(batch_size=128)

fid_value = compute_fid_resnetmini(
    images_01=images,
    classifier=my_model,
    real_loader=real_loader,
    device=device,
    gen_batch_size=128,
    n_real=10000
)

print(f"FID-like (ResNetMini feature space): {fid_value:.4f}")

FID-like (ResNetMini feature space): 4.5786
